In [2]:
import torch
from torch.utils.data import ConcatDataset, DataLoader, Subset
import torchvision
from torchvision import transforms

to_tensor = transforms.ToTensor()

# Load MNIST 1–9
mnist = torchvision.datasets.MNIST(
    root='data/',
    train=True,
    download=True,
    transform=to_tensor,
    target_transform=lambda y: y - 1
)
mnist_mask = mnist.targets >= 1
mnist_1to9 = Subset(mnist, mnist_mask.nonzero(as_tuple=True)[0].tolist())

# Load EMNIST 1–9
emnist = torchvision.datasets.EMNIST(
    root='data/',
    split='digits',
    train=True,
    download=True,
    transform=to_tensor,
    target_transform=lambda y: y - 1
)
emnist_mask = (emnist.targets >= 1) & (emnist.targets <= 9)
emnist_1to9 = Subset(emnist, emnist_mask.nonzero(as_tuple=True)[0].tolist())

# Combine and make a loader
combined = ConcatDataset([mnist_1to9, emnist_1to9])
loader = DataLoader(combined, batch_size=256, shuffle=False, num_workers=0)

# Accumulate sums and squared sums
sum_ = 0.0
sum_sq = 0.0
n_pixels = 0

for imgs, _ in loader:
    # imgs shape: (B, 1, 28, 28)
    B, C, H, W = imgs.shape
    # Sum over all dimensions
    sum_    += imgs.sum().item()
    sum_sq  += (imgs ** 2).sum().item()
    n_pixels += B * C * H * W

# Compute mean and std as scalars
mean = sum_ / n_pixels
var  = (sum_sq / n_pixels) - (mean ** 2)
std  = var ** 0.5

print(f"Computed mean: {mean:.6f}")
print(f"Computed std:  {std:.6f}")

Computed mean: 0.159903
Computed std:  0.323860
